# 02 — Seasonal-Naive Baseline
**Project FORESIGHT — NorthBay Living**

This notebook:
1. Verifies the seasonal-naive baseline runs correctly for a single SKU.
2. Backtests the baseline across all SKUs and computes the headline WAPE.
3. Records the WAPE number that Teammate 2's LightGBM model must beat.

> **Run all cells top-to-bottom after `python -m src.pipeline` has completed.**  
> The WAPE number from Section 3 must be copied into `README.md`.

## Section 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.config import PANEL_PATH, FORECAST_HORIZON_WEEKS
from src.baseline import seasonal_naive_forecast, run_baseline_all_skus
from src.metrics import wape, bias

panel = pd.read_parquet(PANEL_PATH)
panel['date'] = pd.to_datetime(panel['date'])

print(f'Panel loaded: {panel.shape}')
print(f'Date range  : {panel["date"].min().date()} → {panel["date"].max().date()}')
print(f'Unique SKUs : {panel["sku_id"].nunique()}')

## Section 2 — Run Baseline for One SKU (sanity check)

In [ ]:
# Pick the first SKU and run the baseline as a sanity check
as_of_date = pd.Timestamp('2025-10-01')
result = seasonal_naive_forecast(panel, 'SKU0001', as_of_date, horizon_weeks=8)

print('Seasonal-naive forecast for SKU0001 (as of 2025-10-01):')
print(result.to_string(index=False))
print()
print('Checks:')
print(f'  Rows returned : {len(result)}  (expected 8)')
print(f'  Columns       : {result.columns.tolist()}')
print(f'  All yhat >= 0 : {(result["baseline_yhat"] >= 0).all()}')

## Section 3 — Backtest the Baseline on All SKUs

In [ ]:
# One-fold backtest: train up to 2025-10-01, evaluate on the next 8 weeks
as_of_date = pd.Timestamp('2025-10-01')

print('Running baseline for all SKUs...')
baseline_forecasts = run_baseline_all_skus(panel, as_of_date)
print(f'Forecasts generated: {len(baseline_forecasts)} rows, {baseline_forecasts["sku_id"].nunique()} SKUs')

# Get actuals for the same 8-week window
horizon_end = as_of_date + pd.Timedelta(weeks=FORECAST_HORIZON_WEEKS)
actuals = (
    panel[
        (panel['date'] >= as_of_date) &
        (panel['date'] <  horizon_end)
    ]
    .assign(
        week_start=lambda x: x['date'] - pd.to_timedelta(x['date'].dt.dayofweek, unit='D')
    )
    .groupby(['sku_id', 'week_start'])['units_sold']
    .sum()
    .reset_index()
    .rename(columns={'units_sold': 'actual_weekly'})
)
actuals['week_start'] = pd.to_datetime(actuals['week_start'])

# Merge forecasts with actuals
merged = baseline_forecasts.merge(actuals, on=['sku_id', 'week_start'], how='inner')
print(f'Matched rows for evaluation: {len(merged)}')

# Compute metrics
baseline_wape = wape(merged['actual_weekly'].values, merged['baseline_yhat'].values)
baseline_bias = bias(merged['actual_weekly'].values, merged['baseline_yhat'].values)

print()
print('=' * 50)
print('BASELINE PERFORMANCE')
print('=' * 50)
print(f'Baseline WAPE : {baseline_wape:.4f}  ({baseline_wape * 100:.2f}%)')
print(f'Baseline Bias : {baseline_bias:.2f} units/week')
print()
print(f'*** Record this number: Baseline WAPE = {baseline_wape * 100:.2f}% ***')
print('*** Teammate 2 LightGBM model must beat this. ***')

## Section 4 — Record in README.md

Copy the numbers from Section 3 into the `README.md` Results section:

```
## Results
- Baseline (seasonal-naive) WAPE: **[paste number here]%**
- LightGBM forecast WAPE:         **[Teammate 2 fills this in]%**
- Forecast improvement over baseline: **[Teammate 2 fills this in]%**
```

Also message the baseline WAPE number directly to Teammate 2 —  
they need it to know whether their model beats the bar.